> ⚠️ **此 notebook 已废弃（2026-07-04）**  
> 当前项目移除了 `create_agent()` 和中间件，改为手写 StateGraph。等价日志逻辑已内嵌到 `src/agent/graph.py` 的 `agent_node` 中。此 notebook 仅保留作为中间件机制的**教学参考**，代码无法直接运行（`agent_with_middleware` 实例已不存在）。

# 03 — Middleware 中间件

> **目标**：理解中间件的工作机制——在不改核心代码的前提下注入自定义逻辑  
> **产出**：带日志中间件的 Agent，每次 LLM 调用和工具调用都有详细日志

## 0. 准备

In [ ]:
import sys, logging
from pathlib import Path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

# 启用日志，级别设为 INFO 以便看到中间件的输出
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")

from src.config.settings import settings
print(f"模型: {settings.MODEL_NAME}")

## 1. 理解中间件的执行顺序

中间件采用**洋葱模型**（与 FastAPI/Express 一致）：

```
请求进入 → before_agent → before_model → [LLM调用] → after_model → ... → after_agent → 响应
                                        ↑
                          wrap_model_call 包裹这一层
```

多个中间件时：先添加的在外层（先执行 before，后执行 after）。

我们实现的 `RequestLoggingMiddleware` 记录：
- 每次 LLM 调用的开始/结束 + 耗时 + token 用量
- 每次工具调用的名称/参数/结果

## 2. 加载带中间件的 Agent

对比 v2（手写 StateGraph，无中间件）和 v3（create_agent + 中间件）。

In [ ]:
from src.agent.graph import agent_with_middleware

print(f"Agent 类型: {type(agent_with_middleware).__name__}")
print()
# create_agent() 内部会将中间件"编译"进 LangGraph 的节点中，
# 编译后的 graph 上不再暴露 middleware 属性。
# 我们要验证中间件是否生效，直接看运行时的日志输出即可。
print("中间件已编译进 Agent（见下方调用时的日志输出）")

## 3. 调用——观察中间件日志

下面这个调用会触发：
1. `before_model` — 日志：LLM 调用 #1 开始
2. LLM 决定调用 get_current_time 工具
3. `wrap_tool_call` — 日志：工具调用 #1: get_current_time
4. 工具返回结果
5. `after_model` — 日志：LLM 调用 #1 完成（耗时 + token）
6. LLM 再次被调用（如果需要更多工具）或给出最终回答

In [ ]:
result = agent_with_middleware.invoke(
    {"messages": [{"role": "user", "content": "现在几点？"}]}
)

# 打印最终回答
for msg in reversed(result["messages"]):
    if getattr(msg, 'type', '') == "ai" and msg.content:
        print(f"\n>>> 最终回答: {msg.content}")
        break

## 4. 对比：无中间件 vs 有中间件

行为完全一致（Agent 的功能不变），但后者多了可观测性日志。

这就是中间件的核心价值：**横切关注点（日志、监控、权限）从核心业务逻辑中解耦**。

In [ ]:
from src.agent.graph import graph as graph_v2  # 手写版，无中间件

print("=== v2（手写 StateGraph，无中间件）===")
result_v2 = graph_v2.invoke(
    {"messages": [{"role": "user", "content": "1+1=?"}]},
    config={"configurable": {"thread_id": "cmp-1"}},
)
print(f"消息数: {len(result_v2['messages'])} （没有中间件日志输出）\n")

print("=== v3（create_agent + 中间件）===")
result_v3 = agent_with_middleware.invoke(
    {"messages": [{"role": "user", "content": "1+1=?"}]}
)
print(f"消息数: {len(result_v3['messages'])} （上方有中间件日志输出）")

## 5. 中间件还可以做什么？

| 场景 | 中间件 |
|------|--------|
| 敏感信息脱敏 | `PIIMiddleware(patterns=["email", "phone"])` |
| 长对话摘要 | `SummarizationMiddleware(max_tokens_before_summary=500)` |
| 敏感操作审批 | `HumanInTheLoopMiddleware(interrupt_on={...})` |
| 模型故障切换 | `ModelFallbackMiddleware("gpt-4o", "claude-sonnet")` |
| 请求日志 | `RequestLoggingMiddleware()` （本节实现） |
| 自定义权限控制 | 自定义 wrap_tool_call，屏蔽危险操作 |

**下一步**：在 `04_state_graph.ipynb` 中深入 LangGraph 的高级特性。